<a href="https://colab.research.google.com/github/nurfnick/Numerical_Methods/blob/master/Module_4_Guass_Jordan_Elimination/ProjectPart4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Part 4



I don't need lots of the pieces from earlier assignments so I will not include them here.

In [1]:
import math
import numpy as np

## Gauss-Jordan Method

The Gauss-Jordan elimination method is all about solving linear systems of equations.  Something of the form
$$
\left\{
\begin{align}
a_{1,1} x_1 +&\dots a_{1,n} x_n = y_1\\
&\vdots\\
a_{n,1}x_1+&\dots a_{n,n} x_n = y_n
\end{align}
\right.
$$

First thing we need to figure out is how to create an augmented matrix from this!

$$
\left[
\begin{array}{ccc|c}
a_{1,1} & \dots & a_{1,n} & y_1\\
\vdots&\ddots&\vdots&\vdots\\
a_{n,1} &\dots& a_{n,n}&y_n
\end{array}
\right]
$$
And how to deal with this matrix!

In [2]:
A = np.array([[1,2,3],[2,3,4]])

A

array([[1, 2, 3],
       [2, 3, 4]])

In [3]:
def addRows(matrix,targetRow, modifierRow, value):
  newRow = matrix[targetRow,:]+value*matrix[modifierRow,:]
  newMatrix = []
  for i in range(len(matrix)):
    if i != targetRow:
      newMatrix.append(matrix[i,:])
    else:
      newMatrix.append(newRow)
  return np.array(newMatrix)



In [4]:
addRows(A,0,1,10)

array([[21, 32, 43],
       [ 2,  3,  4]])

In [5]:
def swapRows(matrix,row1,row2):
  newMatrix = []
  for i in range(len(matrix)):
    if i == row1:
      newMatrix.append(matrix[row2,:])
    elif i == row2:
      newMatrix.append(matrix[row1,:])
    else:
      newMatrix.append(matrix[i,:])
  return np.array(newMatrix)

#here is another try at this function

def swapRows2(matrix, row1, row2):
    dic = {row1:row2,row2:row1} #create a dictionary that changes row1 to row2 and vise versa
    list = [dic.get(x,x) for x in range(len(matrix))] #apply the dictionary, but return the og if its not in the dic
    return matrix[list] #return the matrix back but with just those two swapped

In [6]:
swapRows(A,0,1)

array([[2, 3, 4],
       [1, 2, 3]])

In [7]:
def changeToLeadingOne(matrix,row,column):
  newMatrix =[]
  for i in range(len(matrix)):
    if i != row:
      newMatrix.append(matrix[i,:])
    else:
      newMatrix.append(1/matrix[row,column]*matrix[row,:])
  return np.array(newMatrix)


In [8]:
changeToLeadingOne(A,1,0)

array([[1. , 2. , 3. ],
       [1. , 1.5, 2. ]])

In [9]:
def gaussJordan(matrix):
  newMatrix = matrix
  for pivot in range(len(matrix)):
    newMatrix = changeToLeadingOne(newMatrix,pivot,pivot)
    for column in range(0,pivot):
      newMatrix = addRows(newMatrix,column,pivot,-newMatrix[column,pivot])
    for column in range(pivot+1,len(matrix)):
      newMatrix = addRows(newMatrix,column,pivot,-newMatrix[column,pivot])
  return newMatrix


In [10]:
gaussJordan(A)

array([[ 1.,  0., -1.],
       [-0.,  1.,  2.]])

Let's try that again with a little more complicated matrix!

In [11]:
B = np.array([[1,2,3,4],[2,3,4,5],[3,4,5,6]])

In [12]:
gaussJordan(B)

/tmp/ipykernel_4964/2247070343.py:7: RuntimeWarning: divide by zero encountered in scalar divide
  newMatrix.append(1/matrix[row,column]*matrix[row,:])
/tmp/ipykernel_4964/2247070343.py:7: RuntimeWarning: invalid value encountered in multiply
  newMatrix.append(1/matrix[row,column]*matrix[row,:])


array([[nan, nan, nan, nan],
       [nan, nan, nan, nan],
       [nan, nan, nan, nan]])

Oh no!  What happened!  Let's follow the process and see where it went wrong.

First we already have a leading 1 so I will somplify the other two

In [13]:
B = addRows(B,1,0,-2)
B

array([[ 1,  2,  3,  4],
       [ 0, -1, -2, -3],
       [ 3,  4,  5,  6]])

In [14]:
B = addRows(B,2,0,-3)
B

array([[ 1,  2,  3,  4],
       [ 0, -1, -2, -3],
       [ 0, -2, -4, -6]])

In [15]:
B = changeToLeadingOne(B,1,1)
B

array([[ 1.,  2.,  3.,  4.],
       [-0.,  1.,  2.,  3.],
       [ 0., -2., -4., -6.]])

New leading one, what could go wrong!

In [16]:
B = addRows(B,0,1,-2)
B

array([[ 1.,  0., -1., -2.],
       [-0.,  1.,  2.,  3.],
       [ 0., -2., -4., -6.]])

In [17]:
addRows(B,2,1,2)

array([[ 1.,  0., -1., -2.],
       [-0.,  1.,  2.,  3.],
       [ 0.,  0.,  0.,  0.]])

Here is the issue!  I get a full compliment of zeros.  While this system is still solvable, my method will fail because I cannot find another pivot (leading 1)  By the way the solution here is that
$$
\begin{align}
x &= -2+t\\
y &= 3-2t\\
z &= t
\end{align}
$$

$z$ the third variable can be what ever it wants but that determines what the rest must me.

##Computing Inverses

I am curious if I can get my algorithm to compute a matrix inverse.  The traditional method to doing this is to take your original square matrix and augment the identity matrix, then preform row ops until you ge the inverse.
$$
\left[A|I\right] \to \left[I|A^{-1}\right]
$$

Let's try it on
$$
A=\left[
\begin{array}{cc}
1&2\\
3&4
\end{array}
\right]
$$


In [18]:
C = np.array([[1,2,1,0],[3,4,0,1]])

In [19]:
gaussJordan(C)

array([[ 1. ,  0. , -2. ,  1. ],
       [-0. ,  1. ,  1.5, -0.5]])

So I think $A^{-1}$ is
$$
A^{-1}=\left[
\begin{array}{cc}
-2&1\\
1.5&-\frac12
\end{array}
\right]
$$
Let's make python check it.  To multiply matricies you use the @ symbol

In [20]:
Ainverse = np.array([[-2,1],[1.5,-.5]])
Anew = np.array([[1,2],[3,4]])

In [21]:
Anew @ Ainverse

array([[1., 0.],
       [0., 1.]])

Sweet!

##Using Some Packages in Python

I haven't used this package yet but scipy is for doing scientific computing in python.  Many of the programs we have created are premade in the scipy package

In [22]:
import scipy.linalg as la

la.inv(Anew)

array([[-2. ,  1. ],
       [ 1.5, -0.5]])

In [23]:
la.solve(A[:,:2],A[:,2])#Using the original A giving it the coeffiecient and then the solutions.

array([-1.,  2.])

# A Fix to My Gauss-Jordan

I've been doing some reading over the break on Linear Algebra and recalled that I did not account for rounding errors in my method.  I think one of the videos even calls out that it is not a very good numerical method if it does not account for this.  To fix this issue, I need to add another step to my algorithm.  I want to make the piviot the largest absolute value.  I'll do this with a simple function that returns what row that happens in.

In [24]:
def findLargestPivot(matrix, column):
  vec = matrix[column:,column]
  vec = np.abs(vec)
  row = np.where(vec == max(vec))
  return row[0][0]+column


In [25]:
vec = np.array([[1,2,-4],[-2,-5,2]])
np.abs(vec)

array([[1, 2, 4],
       [2, 5, 2]])

In [26]:
findLargestPivot(vec,1) #checking that it works even with the negative

np.int64(1)

Okay now that I can find which row should be made the leading one I'll recycle my algorithm adding this step in (and actually using my swap rows function)

In [27]:
def gaussJordan2(matrix):
  newMatrix = matrix
  for pivot in range(len(matrix)):
    newpivot = findLargestPivot(newMatrix,pivot)
    newMatrix = swapRows(newMatrix,newpivot,pivot)
    newMatrix = changeToLeadingOne(newMatrix,pivot,pivot)
    for column in range(0,pivot):
      newMatrix = addRows(newMatrix,column,pivot,-newMatrix[column,pivot])
    for column in range(pivot+1,len(matrix)):
      newMatrix = addRows(newMatrix,column,pivot,-newMatrix[column,pivot])
  return newMatrix

In [28]:
gaussJordan2(vec)

array([[  1.,   0., -16.],
       [ -0.,   1.,   6.]])

This was trickier than I thought it would be.  It is important when looking for the largest pivot you not consider the previous columns that already have the leading ones.  I think **gaussJordan2** does this but clearly I am getting beyond the scope of this class as I try to improve a nice working algorithm.  Just know that my coding is not perfect (and most likely neither will yours).

## LU Decomposition

In [68]:
def LUDecomp(matrix):
  L = np.eye(len(matrix))
  newMatrix = matrix
  for pivot in range(len(matrix)):#No leading 1 pivots
    for column in range(pivot+1,len(matrix)):#Only eliminate below
      L[column,pivot] = newMatrix[column,pivot]/newMatrix[pivot,pivot]#Store the factor before modifying the original matrix
      newMatrix = addRows(newMatrix,column,pivot,-newMatrix[column,pivot]/newMatrix[pivot,pivot])
  return newMatrix,L #Return U and L

In [69]:
C = np.array([
     [4,2,3,4],
     [2,3,4,5],
     [3,4,5,6],
     [1,1,1,2]
])

U, L = LUDecomp(C)

In [70]:
U

array([[ 4.   ,  2.   ,  3.   ,  4.   ],
       [ 0.   ,  2.   ,  2.5  ,  3.   ],
       [ 0.   ,  0.   , -0.375, -0.75 ],
       [ 0.   ,  0.   ,  0.   ,  1.   ]])

In [71]:
L

array([[1.  , 0.  , 0.  , 0.  ],
       [0.5 , 1.  , 0.  , 0.  ],
       [0.75, 1.25, 1.  , 0.  ],
       [0.25, 0.25, 1.  , 1.  ]])

In [73]:
L@U==C

array([[ True,  True,  True,  True],
       [ True,  True,  True,  True],
       [ True,  True,  True,  True],
       [ True,  True,  True,  True]])